# GAIA capstone pipeline with safe submission boundaries

A deterministic capstone scaffold for exact-answer evaluation. It models the official record and payload shapes without including benchmark answers or making live calls.

## Learning objectives

- Model scoring-question and submitted-answer records explicitly.
- Normalize whitespace, numeric forms, and final-answer-only responses for exact match.
- Separate file inspection, calculation, retrieval, search, model, retry, error, and cache boundaries.
- Build the documented submission payload offline while preventing accidental transmission.

## Setup

The complete offline pipeline uses only Python's standard library. Optional HTTP and model cells require `python -m pip install requests smolagents`. Keep each live action disabled until you have inspected its environment guard and configuration. The offline demonstration creates a payload object but performs no request.

## Complete implementation

Run the next cell to define the complete deterministic project pipeline.

In [1]:
import re
from decimal import Decimal, InvalidOperation

# These are synthetic records shaped like the course scoring API. They are not
# GAIA questions or ground-truth answers.
question_records = [
    {
        "task_id": "demo-1",
        "question": "What is six multiplied by seven?",
        "level": 1,
        "file_name": "",
    },
    {
        "task_id": "demo-file",
        "question": "Return the city named in the attached text file.",
        "level": 1,
        "file_name": "demo-city.txt",
    },
]

def normalize_answer(raw_answer: object) -> str:
    """Produce whitespace-trimmed, final-answer-only, stable numeric text."""
    text = " ".join(str(raw_answer).strip().split())
    final_match = re.search(r"(?i)(?:^|\s)final\s+answer\s*:\s*(.+)$", text)
    if final_match:
        text = final_match.group(1).strip()
    numeric_candidate = text.rstrip(".")
    try:
        number = Decimal(numeric_candidate.replace(",", ""))
    except InvalidOperation:
        return text
    if not number.is_finite():
        return text
    normalized = format(number.normalize(), "f")
    return "0" if Decimal(normalized) == 0 else normalized

normalized_examples = [
    normalize_answer(" 42.0 "),
    normalize_answer("  Riga\n"),
    normalize_answer("FINAL ANSWER: 3.1400"),
]

def inspect_demo_file(file_name: str) -> str:
    """Offline file-inspection boundary over an embedded fixture."""
    fixtures = {"demo-city.txt": "The city named in this file is Riga."}
    if file_name not in fixtures:
        raise FileNotFoundError(file_name)
    return fixtures[file_name]

def calculate_offline(expression: str) -> str:
    """Narrow deterministic calculation boundary for the demo question."""
    calculations = {"6 * 7": "42"}
    if expression not in calculations:
        raise ValueError("expression is outside the offline calculation allowlist")
    return calculations[expression]

def retrieve_offline(query: str) -> str:
    """Small local retrieval boundary, distinct from live search."""
    records = {"course exact match": "Submit the answer alone after normalization."}
    return records.get(query.casefold(), "")

def search_disabled_offline(query: str) -> str:
    """Make the offline/live search boundary explicit."""
    raise PermissionError("live search is disabled in the offline pipeline")

offline_adapters = {
    "file_inspector": inspect_demo_file,
    "calculator": calculate_offline,
    "retriever": retrieve_offline,
    "search": search_disabled_offline,
}

def deterministic_solver(question: dict, context: str, adapters: dict) -> str:
    """Solve only the two transparent synthetic demonstrations."""
    if question["task_id"] == "demo-1":
        return adapters["calculator"]("6 * 7")
    if question["task_id"] == "demo-file" and "Riga" in context:
        return "Riga"
    raise ValueError("no deterministic solver route for this synthetic question")

def run_question(
    question: dict,
    *,
    solver,
    adapters: dict,
    cache: dict,
    max_attempts: int = 2,
) -> dict:
    """Run one record with retries, caching, and per-question error capture."""
    if max_attempts < 1:
        raise ValueError("max_attempts must be at least one")
    task_id = question["task_id"]
    if task_id in cache:
        return {
            "task_id": task_id,
            "submitted_answer": cache[task_id],
            "attempts": 0,
            "cached": True,
            "error": None,
        }
    context = ""
    if question.get("file_name"):
        try:
            context = adapters["file_inspector"](question["file_name"])
        except Exception as exc:
            return {
                "task_id": task_id,
                "submitted_answer": "",
                "attempts": 1,
                "cached": False,
                "error": f"{type(exc).__name__}: {exc}",
            }
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            answer = normalize_answer(solver(question, context, adapters))
            if not answer:
                raise ValueError("solver returned an empty normalized answer")
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
        else:
            cache[task_id] = answer
            return {
                "task_id": task_id,
                "submitted_answer": answer,
                "attempts": attempt,
                "cached": False,
                "error": None,
            }
    return {
        "task_id": task_id,
        "submitted_answer": "",
        "attempts": max_attempts,
        "cached": False,
        "error": last_error,
    }

def run_pipeline(questions: list[dict], *, solver, adapters: dict, cache: dict) -> list[dict]:
    """Keep failures local so one question cannot abort the batch."""
    return [
        run_question(
            question,
            solver=solver,
            adapters=adapters,
            cache=cache,
            max_attempts=2,
        )
        for question in questions
    ]

demo_cache = {}
pipeline_results = run_pipeline(
    question_records,
    solver=deterministic_solver,
    adapters=offline_adapters,
    cache=demo_cache,
)
answer_records = [
    {
        "task_id": result["task_id"],
        "submitted_answer": result["submitted_answer"],
    }
    for result in pipeline_results
    if result["error"] is None
]

def build_submission_payload(username: str, agent_code: str, answers: list[dict]) -> dict:
    """Build the documented request body without transmitting it."""
    if not username.strip():
        raise ValueError("username is required")
    if not agent_code.startswith("https://huggingface.co/spaces/") or not agent_code.endswith("/tree/main"):
        raise ValueError("agent_code must be a public Hugging Face Space code URL ending in /tree/main")
    normalized_answers = [
        {
            "task_id": answer["task_id"],
            "submitted_answer": normalize_answer(answer["submitted_answer"]),
        }
        for answer in answers
    ]
    return {
        "username": username.strip(),
        "agent_code": agent_code,
        "answers": normalized_answers,
    }

def validate_live_answer_records(
    live_questions: list[dict], live_answers: list[dict]
) -> list[dict]:
    """Validate a complete live answer set and return it in question order."""
    if not isinstance(live_questions, list) or not live_questions:
        raise ValueError("live question records are undefined or empty")
    if not isinstance(live_answers, list):
        raise ValueError("live answer records are undefined")

    expected_order = []
    expected_ids = set()
    for question in live_questions:
        task_id = str(question.get("task_id", "")).strip()
        if not task_id:
            raise ValueError("live question contains an undefined task ID")
        if task_id.casefold().startswith(("demo", "synthetic", "sample")):
            raise ValueError("synthetic/demo task IDs cannot be submitted")
        if task_id in expected_ids:
            raise ValueError(f"duplicate live question task ID: {task_id}")
        expected_ids.add(task_id)
        expected_order.append(task_id)

    answers_by_id = {}
    for answer in live_answers:
        task_id = str(answer.get("task_id", "")).strip()
        if not task_id:
            raise ValueError("live answer contains an undefined task ID")
        if task_id.casefold().startswith(("demo", "synthetic", "sample")):
            raise ValueError("synthetic/demo task IDs cannot be submitted")
        if task_id in answers_by_id:
            raise ValueError(f"duplicate live answer task ID: {task_id}")
        if task_id not in expected_ids:
            raise ValueError(f"unknown live answer task ID: {task_id}")
        if "submitted_answer" not in answer or answer["submitted_answer"] is None:
            raise ValueError(f"undefined answer for task ID: {task_id}")
        submitted_answer = normalize_answer(answer["submitted_answer"])
        if not submitted_answer:
            raise ValueError(f"empty answer for task ID: {task_id}")
        answers_by_id[task_id] = submitted_answer

    missing_ids = [task_id for task_id in expected_order if task_id not in answers_by_id]
    if missing_ids:
        raise ValueError("missing live answers for task IDs: " + ", ".join(missing_ids))
    return [
        {"task_id": task_id, "submitted_answer": answers_by_id[task_id]}
        for task_id in expected_order
    ]

sample_payload = build_submission_payload(
    "example-user",
    "https://huggingface.co/spaces/example-user/gaia-agent/tree/main",
    [{"task_id": "demo-1", "submitted_answer": "42.0"}],
)

## Deterministic assertions

These assertions verify the important behavior without a network, model, optional package, or credential.

In [2]:
assert normalized_examples == ["42", "Riga", "3.14"]
assert normalize_answer("FINAL ANSWER: 0042.000") == "42"
assert normalize_answer("  New   York  ") == "New York"
assert [result["submitted_answer"] for result in pipeline_results] == ["42", "Riga"]
assert all(result["error"] is None for result in pipeline_results)
assert answer_records == [
    {"task_id": "demo-1", "submitted_answer": "42"},
    {"task_id": "demo-file", "submitted_answer": "Riga"},
]
assert sample_payload == {
    "username": "example-user",
    "agent_code": "https://huggingface.co/spaces/example-user/gaia-agent/tree/main",
    "answers": [{"task_id": "demo-1", "submitted_answer": "42"}],
}
try:
    search_disabled_offline("current fact")
except PermissionError:
    pass
else:
    raise AssertionError("offline search boundary unexpectedly opened")

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

Normalization converts equivalent numeric spellings to stable decimal text, collapses whitespace, and removes a leading `FINAL ANSWER:` label because the scoring API expects the answer alone. Each question owns its retry count and error string, while the cache prevents repeated solver work. File inspection, calculation, retrieval, and search are replaceable adapters. The sample payload has exactly `username`, public-Space `agent_code`, and `answers`, but it remains an in-memory dictionary.

### Exercise

Add a third synthetic question that uses `retrieve_offline`. First write its expected normalized answer and result record, then add a deterministic solver route. Next, inject one controlled failure and verify that the other question records still complete.

### Troubleshooting

If an exact match fails, inspect invisible whitespace and numeric punctuation before changing the solver. If a batch stops early, ensure exceptions are captured inside `run_question`, not around the whole pipeline. If a live cell does anything unexpectedly, clear its opt-in variable immediately; the submission cell also requires the exact explicit-consent phrase.

## Credentials

Offline cells read no credentials and make no HTTP requests. Live GET questions, file download, model call, and POST submission are separate cells with separate guards. The public scoring GET, file, and POST routes do not require or transmit `HF_TOKEN`. The POST cell defaults to skip and requires `ENABLE_LIVE_NETWORK=1`, `ENABLE_GAIA_SUBMIT=I_UNDERSTAND_THIS_SUBMITS`, `HF_USERNAME`, and `GAIA_AGENT_CODE` before importing `requests`. `HF_TOKEN` remains required only for the separate optional model cell.

Unit 1 quiz submission, hosting a public Space, GAIA scoring, and certificate claiming are personal account actions. No ground-truth GAIA answers are included. No score or certification is claimed by this notebook.

In [3]:
import os

enable_live_network = os.getenv("ENABLE_LIVE_NETWORK") == "1"
scoring_api_base_url = os.getenv("GAIA_SCORING_API_BASE_URL", "https://agents-course-unit4-scoring.hf.space")
if not enable_live_network:
    print("SKIPPED: Set ENABLE_LIVE_NETWORK=1 to fetch live scoring questions.")
else:
    try:
        import requests
    except ImportError:
        print("SKIPPED: requests is not installed; run `python -m pip install requests`.")
    else:
        live_questions_response = requests.get(
            scoring_api_base_url.rstrip("/") + "/questions",
            timeout=30,
        )
        live_questions_response.raise_for_status()
        live_question_records = live_questions_response.json()
        # Populate this dedicated collection only with answers for the fetched
        # live question IDs. Offline demo records are never copied into it.
        live_answer_records = []
        print(f"Fetched {len(live_question_records)} live question records; no answers were submitted.")

Fetched 20 live question records; no answers were submitted.


In [4]:
import os

enable_live_network = os.getenv("ENABLE_LIVE_NETWORK") == "1"
enable_file_download = os.getenv("ENABLE_FILE_DOWNLOAD") == "1"
live_task_id = os.getenv("GAIA_TASK_ID")
file_fallback_url = os.getenv("GAIA_FILE_FALLBACK_URL")
scoring_api_base_url = os.getenv("GAIA_SCORING_API_BASE_URL", "https://agents-course-unit4-scoring.hf.space")
if not enable_live_network:
    print("SKIPPED: Set ENABLE_LIVE_NETWORK=1 to download an optional task file.")
elif not enable_file_download:
    print("SKIPPED: Set ENABLE_FILE_DOWNLOAD=1 to download an optional task file.")
elif not live_task_id:
    print("SKIPPED: Set GAIA_TASK_ID to identify the optional task file.")
else:
    try:
        import requests
    except ImportError:
        print("SKIPPED: requests is not installed; run `python -m pip install requests`.")
    else:
        live_file_response = requests.get(
            scoring_api_base_url.rstrip("/") + f"/files/{live_task_id}",
            timeout=30,
        )
        if live_file_response.status_code == 404 and file_fallback_url:
            fallback_response = requests.get(file_fallback_url, timeout=30)
            fallback_response.raise_for_status()
            live_file_bytes = fallback_response.content
            print(
                "Primary file route returned 404; downloaded "
                f"{len(live_file_bytes)} bytes from the configured fallback "
                "into memory; no file was executed."
            )
        else:
            live_file_response.raise_for_status()
            live_file_bytes = live_file_response.content
            print(f"Downloaded {len(live_file_bytes)} bytes into memory; no file was executed.")

Primary file route returned 404; downloaded 63080 bytes from the configured fallback into memory; no file was executed.


In [5]:
import os

enable_model_calls = os.getenv("ENABLE_MODEL_CALLS") == "1"
live_model_provider = os.getenv("LIVE_MODEL_PROVIDER", "huggingface").strip().casefold()
hf_token = os.getenv("HF_TOKEN")
if not enable_model_calls:
    print("SKIPPED: Set ENABLE_MODEL_CALLS=1 to run an optional model call.")
elif live_model_provider != "local" and not hf_token:
    print("SKIPPED: Set HF_TOKEN to run an optional model call.")
else:
    try:
        from smolagents import CodeAgent
        if live_model_provider == "local":
            from smolagents import TransformersModel
        else:
            from smolagents import InferenceClientModel
    except ImportError:
        print("SKIPPED: smolagents is not installed; run `python -m pip install smolagents`.")
    else:
        if live_model_provider == "local":
            live_model = TransformersModel(
                model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
                device_map="cpu",
                torch_dtype="auto",
                max_new_tokens=256,
            )
        else:
            live_model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct", token=hf_token)
        live_capstone_agent = CodeAgent(tools=[], model=live_model, max_steps=3)
        live_raw_answer = live_capstone_agent.run(
            "Solve this synthetic question and output only the final answer: six times seven."
        )
        live_normalized_answer = normalize_answer(live_raw_answer)
        print(live_normalized_answer)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Solve this synthetic question and output only the final answer: six times seven.                                │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-1.5B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = 6 * 7                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: 42

... (7 outputs left out)

[Step 3: Duration 76.52 seconds| Input tokens: 6,447 | Output tokens: 117]

Reached max steps.

[Step 4: Duration 12.66 seconds| Input tokens: 6,872 | Output tokens: 121]

fourty-two


In [ ]:
import os

enable_live_network = os.getenv("ENABLE_LIVE_NETWORK") == "1"
submit_consent = os.getenv("ENABLE_GAIA_SUBMIT")
hf_username = os.getenv("HF_USERNAME")
gaia_agent_code = os.getenv("GAIA_AGENT_CODE")
scoring_api_base_url = os.getenv("GAIA_SCORING_API_BASE_URL", "https://agents-course-unit4-scoring.hf.space")
if not enable_live_network:
    print("SKIPPED: Set ENABLE_LIVE_NETWORK=1 before any live submission.")
elif submit_consent != "I_UNDERSTAND_THIS_SUBMITS":
    print("SKIPPED: Set ENABLE_GAIA_SUBMIT=I_UNDERSTAND_THIS_SUBMITS for an intentional live submission.")
elif not hf_username or not gaia_agent_code:
    print("SKIPPED: Set HF_USERNAME and GAIA_AGENT_CODE for an intentional live submission.")
elif "live_question_records" not in globals() or "live_answer_records" not in globals():
    print("REFUSED: Fetch live_question_records and populate the dedicated live_answer_records collection first.")
else:
    try:
        validated_live_answers = validate_live_answer_records(
            live_question_records, live_answer_records
        )
    except ValueError as validation_error:
        print(f"REFUSED: {validation_error}")
    else:
        try:
            import requests
        except ImportError:
            print("SKIPPED: requests is not installed; run `python -m pip install requests`.")
        else:
            live_submission_payload = build_submission_payload(
                hf_username, gaia_agent_code, validated_live_answers
            )
            live_submit_response = requests.post(
                scoring_api_base_url.rstrip("/") + "/submit",
                json=live_submission_payload,
                timeout=60,
            )
            live_submit_response.raise_for_status()
            print(live_submit_response.json())

## Completion checklist

- [ ] Inspected the official-shaped question and answer records.
- [ ] Ran normalization, adapters, retries, caching, and error capture offline.
- [ ] Verified the exact sample payload without sending it.
- [ ] Reviewed all live guards and the additional submission-consent phrase.
- [ ] Completed personal account actions only in my own authenticated session.

## Attribution

The synthetic questions, adapters, retry/cache pipeline, and normalization examples are original and contain no benchmark ground truth. Endpoint names and payload fields follow the official [Hugging Face Agents Course Unit 4 hands-on guide](https://huggingface.co/learn/agents-course/en/unit4/hands-on). The optional agent shape follows the official [smolagents agent reference](https://huggingface.co/docs/smolagents/reference/agents). Live actions were not performed while generating this notebook. Notebook metadata carries the Apache-2.0 license.